## 1. 설정

In [47]:
import os
import re
import json
import glob
from pathlib import Path

import fitz  # PyMuPDF
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

# ----- 설정값: 필요에 맞게 수정하세요 -----
PDF_DIR = "data/"   # 법령 PDF들이 들어있는 폴더 (여러 개)

CHAT_MODEL = "gpt-4o"
EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 3072

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "lawdb")

openai_client = OpenAI()
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

pdf_paths = sorted(glob.glob(str(Path(PDF_DIR) / "*.pdf")))
print(f"발견된 PDF {len(pdf_paths)}개")
for p in pdf_paths:
    print(" -", p)

발견된 PDF 16개
 - data\공무원보수규정(대통령령)(제36501호)(20260707).pdf
 - data\공무원수당 등에 관한 규정(대통령령)(제36015호)(20260102).pdf
 - data\군인 재해보상법(법률)(제20640호)(20250708).pdf
 - data\군인 징계령 시행규칙(국방부령)(제01203호)(20260128).pdf
 - data\군인보수법(법률)(제20802호)(20260319).pdf
 - data\군인복지기본법(법률)(제20638호)(20250708).pdf
 - data\군인사법 시행령(대통령령)(제36067호)(20260203).pdf
 - data\군인사법(법률)(제21319호)(20260203).pdf
 - data\군인연금법(법률)(제21065호)(20251001).pdf
 - data\군인의 지위 및 복무에 관한 기본법 시행령(대통령령)(제35925호)(20260108).pdf
 - data\군인의 지위 및 복무에 관한 기본법(법률)(제20641호)(20260108).pdf
 - data\군형법(법률)(제18465호)(20220701).pdf
 - data\병역법 시행령(대통령령)(제35948호)(20260102).pdf
 - data\병역법(법률)(제21769호)(20260609).pdf
 - data\부대관리훈령(국방부훈령)(제3148호)(20260319).pdf
 - data\제대군인지원에 관한 법률(법률)(제19525호)(20240112).pdf


## 2. PDF 원문 텍스트 추출

In [ ]:
def extract_full_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages_text = [page.get_text("text") for page in doc]
    doc.close()
    return "\n".join(pages_text)

# print(extract_full_text(pdf_paths[1])[1000:2000])

나 업무 수행을 위하여 국외의 교육연구기관, 국
제기구 또는 외국정부기관에 파견(「공무원 인재개발법」에 따른 파견 및 「군위탁생규정」 제4조에 따른 외국유학을
위한 파견은 제외한다)되거나 국가적 사업의 수행을 위하여 특히 필요하여 「공공기관의 운영에 관한 법률」제4조제
1항 각 호의 규정에 의한 공공기관 등의 국외지사에 파견된 공무원 및 「재외국민의 교육지원 등에 관한 법률 시행령
」 제15조제1항에 따라 국외에 파견된 공무원에게는 재외공관에 근무하는 공무원(재외공관에 주재하는 무관을 포함
하며, 이하 “재외공무원”이라 한다)에게 지급하는 수당등에 관한 규정을 준용하여 그 규정에 따른 수당등을 지급한다
. <개정 2016. 1. 8.>
② 제1항의 경우 수당등의 지급 시 계급을 적용할  때는 「공무원임용령」 제16조제1항제3호에서 정한 임용예정계급
상당경력기준의 계급에 따른다.
[전문개정 2008. 12. 31.]
 
       제2장 상여수당 <개정 1993. 12. 31.>
 
제5조 삭제 <1993. 12. 31.>
 
제5조의2 삭제 <1991. 12. 31.>
 
제6조 삭제 <2006. 1. 12.>
 
제6조의2(대우공무원수당) ① 「공무원임용령」 제35조의4, 「군무원인사법 시행령」 제44조,「경찰공무원 승진임용 규정
」 제43조, 「해양경찰청 소속 경찰공무원 임용에 관한 규정」 제93조, 「소방공무원 승진임용 규정」 제43조에 따라 대
우공무원으로 선발된 사람에게는 예산의 범위에서 해당 공무원 월봉급액의 4.1퍼센트를 대우공무원수당으로 지급할
수 있다. 다만, 대우공무원수당과 월봉급액을 합산한 금액이 상위직급으로 승진 시의 월봉급액을 초과할 경우에는

법제처                                                            2                                                       국가법령정보센터
공무원수당 등에 관한 규정
해당 직급 월봉급액과 상위 직급 월봉급액

## 3. 법령 메타데이터 추출 (GPT)


In [33]:
def extract_law_metadata(pdf_path: str, full_text: str) -> dict:
    excerpt = full_text[:3000]  # 법령명, 공포/시행일은 보통 앞부분에 있음

    system_prompt = """당신은 법령 문서에서 메타데이터를 추출하는 전문가입니다.
주어진 텍스트에서 아래 필드를 JSON으로 추출하세요. 모르면 빈 문자열로 두세요.

{
  "name": "법령명 (예: 군인사법)",
  "law_type": "법률 | 대통령령 | 총리령 | 부령 | 훈령 | 예규 | 고시 중 하나",
  "promulgation_date": "공포일 (YYYY-MM-DD 형식, 모르면 빈 문자열)",
  "effective_date": "시행일 (YYYY-MM-DD 형식, 모르면 빈 문자열)"
}

오직 JSON만 출력하세요. 설명이나 코드블록 표시 없이 순수 JSON 텍스트만 출력하세요."""

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": excerpt},
        ],
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r'^```(json)?\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()

    try:
        meta = json.loads(raw)
    except json.JSONDecodeError:
        meta = {"name": Path(pdf_path).stem, "law_type": "", "promulgation_date": "", "effective_date": ""}

    if not meta.get("name"):
        meta["name"] = Path(pdf_path).stem

    # 공백/개행 등으로 인한 문자열 불일치(law_id 매칭 실패)를 막기 위해 모든 필드를 트리밍
    for key in ("name", "law_type", "promulgation_date", "effective_date"):
        if isinstance(meta.get(key), str):
            meta[key] = meta[key].strip()

    return meta


In [ ]:
# extract_law_metadata(pdf_paths[1], extract_full_text(pdf_paths[1]))

{'name': '공무원수당 등에 관한 규정',
 'law_type': '대통령령',
 'promulgation_date': '2026-01-02',
 'effective_date': '2026-01-02'}

## 4. 조문 파싱 + 조번호 정규화

In [59]:
import re

# 실제 조문 헤더는 거의 항상 "줄 맨 앞"에서 시작합니다 (예: "제31조(직권면직)").
# 본문 안에 등장하는 인용구(예: "「장애인복지법」제2조제2항에 따른")는 문장 중간에 있으므로
# (?:^|\n) 로 "줄 시작 위치"인 경우만 진짜 조문 헤더로 인정합니다.
ARTICLE_PATTERN = re.compile(
    r'(?:^|\n)\s*제(\d+)조(?:의(\d+))?\s*(\([^)]{1,40}\))?',
    re.MULTILINE,
)

# 부칙 시작 지점을 찾기 위한 패턴
# 줄 시작 부분에 "부 칙" 또는 "[부칙]" 등이 단독으로 나오는 경우를 매칭합니다.
ADDENDA_PATTERN = re.compile(r'(?:^|\n)\s*(?:부\s*칙|\[부칙\])', re.MULTILINE)


def normalize_article_id(num: str, sub: str = None) -> str:
    return f"{num}-{sub}" if sub else num


def build_original_id(num: str, sub: str = None) -> str:
    return f"제{num}조의{sub}" if sub else f"제{num}조"


def parse_articles(full_text: str) -> list[dict]:
    """본문 텍스트를 조문 단위로 분리. 부칙 이후의 내용은 제외함."""
    
    # 1. 부칙 위치를 찾아 그 뒤는 잘라내기
    addenda_match = ADDENDA_PATTERN.search(full_text)
    if addenda_match:
        # 부칙이 시작되는 index 전까지만 text로 사용
        full_text = full_text[:addenda_match.start()]

    # 2. 본문 조문 파싱 진행
    matches = list(ARTICLE_PATTERN.finditer(full_text))
    articles = []

    for i, m in enumerate(matches):
        num, sub, title_paren = m.group(1), m.group(2), m.group(3)

        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        content = full_text[start:end].strip()

        # 너무 짧은 내용(목차의 조문 나열, 오탐 등)은 제외
        if len(content) < 10:
            continue

        title = title_paren.strip("()") if title_paren else ""

        articles.append({
            "original_id": build_original_id(num, sub),
            "original_id_normalized": normalize_article_id(num, sub),
            "name": title,
            "content": content,
        })

    return articles

In [62]:
print(len(parse_articles(extract_full_text(pdf_paths[2]))))
parse_articles(extract_full_text(pdf_paths[2]))

64


[{'original_id': '제1조',
  'original_id_normalized': '1',
  'name': '목적',
  'content': '이 법은 군인의 공무로 인한 부상ㆍ질병ㆍ장해ㆍ사망에 대하여 적합한 보상을 함으로써 군인 및 그 유족의\n복지 향상에 이바지함을 목적으로 한다.'},
 {'original_id': '제2조',
  'original_id_normalized': '2',
  'name': '적용 범위',
  'content': '이 법은 현역 또는 소집되어 군에 복무하는 군인에게 적용한다. 다만, 다음 각 호의 어느 하나에 해당\n하는 사람(이하 “지원에 의하지 아니하고 임용된 부사관등”이라 한다)에게는 제33조에 따른 장애보상금과 제39조에\n따른 사망보상금만 적용한다.\n1. 지원에 의하지 아니하고 임용된 부사관\n2. 병(兵)\n3. 군간부후보생. 다만, 준사관 또는 부사관(제1호의 부사관은 제외한다)으로 복무 중에 군간부후보생에 지원한 사\n람은 제외한다.'},
 {'original_id': '제3조',
  'original_id_normalized': '3',
  'name': '정의',
  'content': '이 법에서 사용하는 용어의 뜻은 다음과 같다.\n1. “공무상 재해”란 적과의 교전이나 무장폭동 또는 반란을 진압하기 위한 직무, 생명과 신체에 대한 고도의 위험을\n무릅쓴 직무 등을 포함한 군인의 공무(公務)상 부상 또는 공무상 질병이나 그로 인한 장해 및 공무상 사망을 말한\n다.\n2. “유족”이란 군인 또는 군인이었던 사람의 사망 당시 그가 부양하고 있던 「군인연금법」 제3조제1항제4호 및 제\n2항부터 제4항까지의 규정에 해당하는 사람을 말한다. 다만, 제39조에 따른 사망보상금을 지급하는 경우에는 부\n양 여부를 가리지 아니하고 유족으로 본다.\n3. “치유”란 부상 또는 질병이 완치되거나 치료의 효과를 더 이상 기대할 수 없고 그 증상이 고정된 상태에 이르게\n된 것을 말한다.\n4. “장해”란 부

## 5. 조문 인용관계 추출

In [63]:
CROSS_LAW_REF_PATTERN = re.compile(r'「([^」]{1,30})」\s*제(\d+)조(?:의(\d+))?')
SAME_LAW_REF_PATTERN = re.compile(r'제(\d+)조(?:의(\d+))?')


def extract_references(content: str, self_original_id: str, self_law_name: str) -> list[dict]:
    """조문 본문에서 인용된 다른 조문들을 추출. 각 항목은 dict(law_name, original_id_normalized)"""
    refs = []
    seen = set()
    self_law_name = self_law_name.strip()

    # 1) 다른 법령을 명시적으로 인용하는 경우: 「군인사법」 제7조
    for m in CROSS_LAW_REF_PATTERN.finditer(content):
        law_name, num, sub = m.group(1), m.group(2), m.group(3)
        law_name = law_name.strip()  # PDF 추출 시 붙는 trailing space 등을 제거 (매칭 실패 방지)
        norm_id = normalize_article_id(num, sub)
        key = (law_name, norm_id)
        if key in seen:
            continue
        seen.add(key)
        refs.append({"law_name": law_name, "original_id_normalized": norm_id})

    # 2) 법령명 없이 조번호만 언급 (같은 법령 내 인용으로 간주)
    #    단, 위에서 이미 명시적으로 잡힌 「법령」제N조 뒤의 제N조 부분은 제외하기 위해
    #    cross-law 패턴이 매치된 구간을 마스킹한 뒤 탐색
    masked = CROSS_LAW_REF_PATTERN.sub(lambda mm: " " * len(mm.group(0)), content)

    for m in SAME_LAW_REF_PATTERN.finditer(masked):
        num, sub = m.group(1), m.group(2)
        norm_id = normalize_article_id(num, sub)
        if norm_id == self_original_id:
            continue  # 자기 자신 인용은 제외
        key = (self_law_name, norm_id)
        if key in seen:
            continue
        seen.add(key)
        refs.append({"law_name": self_law_name, "original_id_normalized": norm_id})

    return refs


## 6. 임베딩 생성

In [64]:
def embed_texts(texts: list[str], batch_size: int = 100) -> list[list[float]]:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_embeddings.extend([d.embedding for d in response.data])
    return all_embeddings


## 7. Neo4j 스키마 준비 (제약조건 + 벡터 인덱스)

In [65]:
def setup_schema():
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("CREATE CONSTRAINT law_id IF NOT EXISTS FOR (l:LAW) REQUIRE l.id IS UNIQUE")
        session.run("CREATE CONSTRAINT article_id IF NOT EXISTS FOR (a:ARTICLE) REQUIRE a.id IS UNIQUE")

        session.run(f"""
        CREATE VECTOR INDEX article_vector_index IF NOT EXISTS
        FOR (a:ARTICLE) ON (a.embedding)
        OPTIONS {{
          indexConfig: {{
            `vector.dimensions`: {EMBED_DIM},
            `vector.similarity_function`: 'cosine'
          }}
        }}
        """)
    print("스키마(제약조건 + 벡터 인덱스) 준비 완료")


setup_schema()


스키마(제약조건 + 벡터 인덱스) 준비 완료


## 8. 단일 PDF 처리 (파싱 → 임베딩 → 적재 준비)

In [66]:
def process_pdf(pdf_path: str) -> dict:
    full_text = extract_full_text(pdf_path)
    meta = extract_law_metadata(pdf_path, full_text)
    law_id = meta["name"]  # 법령명을 LAW의 고유 id로 사용

    raw_articles = parse_articles(full_text)

    for art in raw_articles:
        art["references"] = extract_references(
            art["content"], art["original_id_normalized"], law_id
        )
        art["id"] = f"{law_id}::{art['original_id']}"
        art["law_id"] = law_id
        art["source_pdf"] = Path(pdf_path).name

    print(f"[{law_id}] 조문 {len(raw_articles)}개 파싱 완료 → 임베딩 생성 중...")
    contents = [a["content"] for a in raw_articles]
    if contents:
        embeddings = embed_texts(contents)
        for art, emb in zip(raw_articles, embeddings):
            art["embedding"] = emb

    return {
        "law": {"id": law_id, **meta},
        "articles": raw_articles,
    }


## 9. Neo4j 적재

모든 PDF를 처리한 결과를 받아서:
1. LAW 노드 MERGE
2. ARTICLE 노드 MERGE (embedding 포함) + LAW-[:CONTAINS]->ARTICLE
3. 모든 조문이 적재된 후, REFERENCES 관계를 조번호 매칭으로 생성 (대상이 없으면 스킵)

In [68]:
def load_law_and_articles(session, law_data: dict):
    law = law_data["law"]
    session.run("""
    MERGE (l:LAW {id: $id})
    SET l.name = $name,
        l.law_type = $law_type,
        l.promulgation_date = $promulgation_date,
        l.effective_date = $effective_date
    """, **law)

    for art in law_data["articles"]:
        session.run("""
        MATCH (l:LAW {id: $law_id})
        MERGE (a:ARTICLE {id: $id})
        SET a.law_id = $law_id,
            a.name = $name,
            a.description = $content,
            a.original_id = $original_id,
            a.original_id_normalized = $original_id_normalized,
            a.source_pdf = $source_pdf,
            a.embedding = $embedding
        MERGE (l)-[:CONTAINS]->(a)
        """, **{k: art[k] for k in [
            "id", "law_id", "name", "content", "original_id",
            "original_id_normalized", "source_pdf", "embedding"
        ]})


def load_references(session, all_law_data: list[dict], verbose: bool = True):
    """인용관계를 연결한다. 디버깅을 위해 시도/성공/실패 건수와
    실패한 인용 대상 샘플을 같이 출력한다."""
    attempted = 0
    matched = 0
    unmatched_samples = []

    for law_data in all_law_data:
        for art in law_data["articles"]:
            for ref in art["references"]:
                attempted += 1
                result = session.run("""
                MATCH (src:ARTICLE {id: $src_id})
                MATCH (tgt:ARTICLE {law_id: $tgt_law_id, original_id_normalized: $tgt_norm_id})
                MERGE (src)-[:REFERENCES]->(tgt)
                RETURN count(*) AS c
                """,
                    src_id=art["id"],
                    tgt_law_id=ref["law_name"],
                    tgt_norm_id=ref["original_id_normalized"],
                )
                c = result.single()["c"]
                matched += c
                if c == 0 and len(unmatched_samples) < 20:
                    unmatched_samples.append({
                        "from": art["id"],
                        "target_law": ref["law_name"],
                        "target_norm_id": ref["original_id_normalized"],
                    })

    print(f"인용 시도: {attempted}건 / 실제 연결: {matched}건 / 매칭 실패: {attempted - matched}건")

    if verbose and unmatched_samples:
        print("\n매칭 실패 샘플 (최대 20개):")
        for s in unmatched_samples:
            print(f'  {s["from"]}  ->  law="{s["target_law"]}" norm_id="{s["target_norm_id"]}" (해당 조문을 DB에서 못 찾음)')

    return {"attempted": attempted, "matched": matched}


## 10. 전체 파이프라인 실행 (여러 PDF)

In [69]:
def run_pipeline(pdf_paths: list[str]):
    all_law_data = []
    for pdf_path in tqdm(pdf_paths, desc="PDF 처리 중"):
        law_data = process_pdf(pdf_path)
        all_law_data.append(law_data)

    print("\nNeo4j에 적재 중...")
    with driver.session(database=NEO4J_DATABASE) as session:
        for law_data in all_law_data:
            load_law_and_articles(session, law_data)
        # 모든 조문이 들어간 뒤에 인용관계 연결 (파일 간 상호 참조 지원)
        load_references(session, all_law_data)

    total_articles = sum(len(ld["articles"]) for ld in all_law_data)
    print(f"\n완료: 법령 {len(all_law_data)}개 / 조문 {total_articles}개 적재")
    return all_law_data


all_law_data = run_pipeline(pdf_paths)

PDF 처리 중:   0%|          | 0/16 [00:00<?, ?it/s]

[공무원보수규정] 조문 94개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중:   6%|▋         | 1/16 [00:06<01:35,  6.35s/it]

[공무원수당 등에 관한 규정] 조문 44개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중:  12%|█▎        | 2/16 [00:09<01:01,  4.40s/it]

...

[부대관리훈령] 조문 412개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중:  94%|█████████▍| 15/16 [00:51<00:04,  4.91s/it]

[제대군인지원에 관한 법률] 조문 53개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중: 100%|██████████| 16/16 [00:53<00:00,  3.32s/it]



Neo4j에 적재 중...
인용 시도: 2744건 / 실제 연결: 1895건 / 매칭 실패: 849건

매칭 실패 샘플 (최대 20개):
  공무원보수규정::제3조  ->  law="공공기관의 운영에 관한 법률" norm_id="4" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제4조의2  ->  law="공무원 수당 등에 관한 규정" norm_id="7-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제4조의2  ->  law="공무원 수당 등에 관한 규정" norm_id="23" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제6조  ->  law="공무원임용령" norm_id="29" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="국가공무원법" norm_id="26-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무원법" norm_id="25-3" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="공무원임용령" norm_id="3-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무원 임용령" norm_id="3-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="공무원임용령" norm_id="3-3" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지
방공무원 임용령" norm_id="3-5" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="국가공무원법" norm_id="71" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무원법" norm_id="63" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="공무원임용령" norm_id="57-3" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무

## 11. 검증

적재가 잘 됐는지 확인하는 몇 가지 쿼리입니다.

In [70]:
def verify():
    with driver.session(database=NEO4J_DATABASE) as session:
        law_count = session.run("MATCH (l:LAW) RETURN count(l) AS c").single()["c"]
        article_count = session.run("MATCH (a:ARTICLE) RETURN count(a) AS c").single()["c"]
        ref_count = session.run("MATCH ()-[r:REFERENCES]->() RETURN count(r) AS c").single()["c"]
        print(f"LAW 노드: {law_count}개")
        print(f"ARTICLE 노드: {article_count}개")
        print(f"REFERENCES 관계: {ref_count}개")

        print("\n샘플 조문 5개:")
        results = session.run("MATCH (a:ARTICLE) RETURN a.id AS id, a.original_id AS oid LIMIT 5")
        for r in results:
            print(" -", r["id"])


verify()


LAW 노드: 16개
ARTICLE 노드: 1697개
REFERENCES 관계: 1849개

샘플 조문 5개:
 - 공무원보수규정::제1조
 - 공무원보수규정::제2조
 - 공무원보수규정::제3조
 - 공무원보수규정::제3조의2
 - 공무원보수규정::제3조의3


In [ ]:
driver.close()